# FastMCP Server with Hugging Face Hub Resources

Build a FastMCP server that exposes Hugging Face Hub models and datasets as queryable MCP resources.

[Read the full article](https://jheiduk.com/posts/fastmcp-huggingface-hub/)

In [ ]:
# MCP framework and Hugging Face Hub client
!pip install "fastmcp>=2.0" huggingface_hub

## Querying Hugging Face Hub API

These cells demonstrate the raw `HfApi` calls that the MCP server wraps as resources. Run them standalone to inspect the data shape before adding the MCP layer.

In [ ]:
from huggingface_hub import HfApi
import json

api = HfApi()

In [ ]:
# List top 5 trending text-generation models
models = list(api.list_models(task="text-generation", sort="trending", limit=5))

for m in models:
    print(f"{m.id}: {m.likes or 0} likes, {m.downloads or 0} downloads")

In [ ]:
# Fetch full metadata for a specific model
info = api.model_info("google/gemma-2-2b")

print(json.dumps({
    "id": info.id,
    "author": info.author,
    "task": info.pipeline_tag or "unknown",
    "likes": info.likes,
    "downloads": info.downloads,
    "tags": info.tags[:15],  # cap to avoid oversized output
}, indent=2))

In [ ]:
# List top 5 trending datasets
datasets = list(api.list_datasets(sort="trending", limit=5))

for d in datasets:
    print(f"{d.id}: {d.likes or 0} likes, {d.downloads or 0} downloads")

In [ ]:
# Fetch full metadata for a specific dataset
info = api.dataset_info("rajpurkar/squad")

print(json.dumps({
    "id": info.id,
    "author": info.author,
    "likes": info.likes,
    "downloads": info.downloads,
    "tags": info.tags[:15],
}, indent=2))

## Building the FastMCP Server

This cell writes `hf_server.py` to disk — the complete FastMCP server with four resources: two static listing resources and two URI-template resources for individual model and dataset lookup.

In [ ]:
server_code = '''
from fastmcp import FastMCP
from huggingface_hub import HfApi
import json

mcp = FastMCP("HuggingFace Hub Explorer")
api = HfApi()


@mcp.resource("hf://models")
def list_models() -> str:
    """Top 10 trending text-generation models on Hugging Face Hub."""
    models = list(api.list_models(task="text-generation", sort="trending", limit=10))
    return json.dumps([
        {"id": m.id, "likes": m.likes or 0, "downloads": m.downloads or 0}
        for m in models
    ], indent=2)


@mcp.resource("hf://datasets")
def list_datasets() -> str:
    """Top 10 trending datasets on Hugging Face Hub."""
    datasets = list(api.list_datasets(sort="trending", limit=10))
    return json.dumps([
        {"id": d.id, "likes": d.likes or 0, "downloads": d.downloads or 0}
        for d in datasets
    ], indent=2)


@mcp.resource("hf://models/{owner}/{name}")
def get_model(owner: str, name: str) -> str:
    """Fetch metadata for a specific model from Hugging Face Hub."""
    info = api.model_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "task": info.pipeline_tag or "unknown",
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


@mcp.resource("hf://datasets/{owner}/{name}")
def get_dataset(owner: str, name: str) -> str:
    """Fetch metadata for a specific dataset from Hugging Face Hub."""
    info = api.dataset_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("hf_server.py", "w") as f:
    f.write(server_code.strip())

print("Server written to hf_server.py")

## Testing with the FastMCP Client

`Client("hf_server.py")` launches the server as a subprocess over stdio. The cells below write and execute a test client that reads all four resources. Make sure the server was written in the cell above before running.

In [ ]:
client_code = '''
import asyncio
from fastmcp import Client


async def main():
    async with Client("hf_server.py") as client:
        # 1. Trending model catalogue
        result = await client.read_resource("hf://models")
        print("Trending models (first 400 chars):")
        print(result[0].text[:400])

        # 2. Specific model detail via URI template
        result = await client.read_resource("hf://models/google/gemma-2-2b")
        print("\nModel detail — google/gemma-2-2b:")
        print(result[0].text)

        # 3. Trending dataset catalogue
        result = await client.read_resource("hf://datasets")
        print("\nTrending datasets (first 400 chars):")
        print(result[0].text[:400])

        # 4. Specific dataset detail via URI template
        result = await client.read_resource("hf://datasets/rajpurkar/squad")
        print("\nDataset detail — rajpurkar/squad:")
        print(result[0].text)


asyncio.run(main())
'''

with open("hf_client.py", "w") as f:
    f.write(client_code.strip())

print("Client written to hf_client.py")

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "hf_client.py"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)